# Immune integration: ATAC tile loading

Load ATAC fragment files for all 99 samples, create 1kb tile count matrix using SnapATAC2/cell2state.

**Environment**: `cell2state_v2026_cuda124_torch25` (NOT regularizedvi)
**Input**: `results/immune_integration/adata_rna.h5ad` (for cell barcodes + fragment_file_path)
**Output**: `results/immune_integration/adata_atac_tiles.h5ad`

In [ ]:
import scanpy as sc
import numpy as np
import os

## Load RNA metadata

In [ ]:
data_folder = "results/immune_integration/"
rna_path = os.path.join(data_folder, "adata_rna.h5ad")

adata = sc.read_h5ad(rna_path)
print(f"RNA adata: {adata.shape}")
print(f"Batches: {adata.obs['batch'].nunique()}")
print(f"fragment_file_path column: {'fragment_file_path' in adata.obs.columns}")

# Check all fragment file paths exist
if "fragment_file_path" in adata.obs.columns:
    paths = adata.obs["fragment_file_path"].unique()
    missing = [p for p in paths if not os.path.exists(p)]
    print(f"\nFragment files: {len(paths)} unique paths")
    if missing:
        print(f"WARNING: {len(missing)} missing files:")
        for p in missing[:5]:
            print(f"  {p}")
    else:
        print("All fragment files exist")

## Rename barcodes for cell2state compatibility

cell2state's `load_atac()` constructs obs_names as `<sample>-<raw_barcode>`. Rename RNA barcodes to match, including UUID mapping for TEA-seq/multiome fragment files.

In [ ]:
from data_loading_utils import rename_obs_for_cell2state

adata = rename_obs_for_cell2state(adata)

## Check SnapATAC2 cached files

In [ ]:
# Check cached h5ad files next to fragment files
cached_files = []
missing_cache = []

for path in sorted(adata.obs["fragment_file_path"].unique()):
    h5ad_cache = path.replace(".tsv.gz", ".h5ad")
    if os.path.exists(h5ad_cache):
        cached_files.append(h5ad_cache)
    else:
        missing_cache.append(path)

print(f"Cached h5ad files: {len(cached_files)}/{len(cached_files) + len(missing_cache)}")
if missing_cache:
    print(f"\nMissing cache for {len(missing_cache)} samples:")
    for p in missing_cache[:5]:
        print(f"  {p}")

## Load ATAC tiles

In [ ]:
# Papermill parameters
genome_ref = "/nfs/srpipe_references/downloaded_from_10X/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/"
max_frag_size_split = 160
bin_size = 1000
counting_strategy = "insertion"
results_folder = "results/immune_integration/"

In [ ]:
from cell2state.utils.aggregation_v2 import concatenate_h5ad

adata_atac = concatenate_h5ad(
    adata,
    variable_type="atac_tiles",
    batch_key="batch",
    loading_kwargs={
        "path_to_reference": genome_ref,
        "path_to_fragment_file_key": "fragment_file_path",
        "max_frag_size_split": max_frag_size_split,
        "bin_size": bin_size,
        "counting_strategy": counting_strategy,
        "use_complete_path": True,
    },
)

print(f"Concatenated ATAC tiles: {adata_atac.shape}")
print(f"dtype: {adata_atac.X.dtype}")

## Post-processing

In [ ]:
# Verify uint16 dtype and add counts layer
import scipy.sparse as sp

if not sp.issparse(adata_atac.X):
    adata_atac.X = sp.csr_matrix(adata_atac.X)
if adata_atac.X.dtype != np.uint16:
    print(f"WARNING: dtype is {adata_atac.X.dtype}, casting to uint16")
    adata_atac.X = adata_atac.X.astype(np.uint16)

# Store counts layer
adata_atac.layers["counts"] = adata_atac.X.copy()

print(f"Final shape: {adata_atac.shape}")
print(f"dtype: {adata_atac.X.dtype}")
print(f"Memory: {adata_atac.X.data.nbytes / 1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt

# ATAC QC metrics
total_frags = np.array(adata_atac.X.sum(1)).squeeze()
n_tiles = np.array((adata_atac.X > 0).sum(1)).squeeze()

adata_atac.obs["total_fragments"] = total_frags
adata_atac.obs["n_tiles_accessible"] = n_tiles

print(f"Total fragments: median={np.median(total_frags):.0f}, range=[{total_frags.min()}, {total_frags.max()}]")
print(f"Tiles accessible: median={np.median(n_tiles):.0f}, range=[{n_tiles.min()}, {n_tiles.max()}]")

# Save ATAC QC metrics as CSV (for use by RNA training notebook)
atac_qc_df = adata_atac.obs[["total_fragments", "n_tiles_accessible"]].copy()
atac_qc_df.to_csv(os.path.join(results_folder, "atac_qc_metrics.csv"))
print(f"Saved ATAC QC metrics: {os.path.join(results_folder, 'atac_qc_metrics.csv')}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(np.log10(total_frags + 1), bins=100)
axes[0].set_xlabel("log10(total fragments)")
axes[0].set_title("ATAC fragments per cell")
axes[1].hist(np.log10(n_tiles + 1), bins=100)
axes[1].set_xlabel("log10(tiles accessible)")
axes[1].set_title("Accessible tiles per cell")
plt.tight_layout()
plt.show()

## Save

In [ ]:
output_path = os.path.join(results_folder, "adata_atac_tiles.h5ad")
adata_atac.write_h5ad(output_path)
print(f"Saved to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1e9:.1f} GB")